In [1]:
from pathlib import Path
from metasmith.agents import Agent
from metasmith.models.libraries import *
from metasmith.models.remote import *

from local.constants import WORKSPACE_ROOT
SOCKEYE_SOURCE = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=64a5c402-05c4-4607-bbad-46a9c2aebd98&origin_path=%2Fhome%2Ftxyliu%2F")
SOCKEYE_SOURCE.endpoint

'64a5c402-05c4-4607-bbad-46a9c2aebd98'

In [2]:
agent_local = Agent(
    home=Source.FromLocal(WORKSPACE_ROOT/"main/local_mock/cache/local_home"),
)

agent_ssh = Agent(
    home=SshSource(
        host="cosmos",
        path="~/workspace/metasmith_home",
    ).AsSource(),
)

agent_slurm = Agent(
    setup_commands=[
        "module load gcc/9.4.0 apptainer/1.3.1",
    ],
    home=SshSource(
        host="sockeye",
        path="~/scratch/metasmith_home",
    ).AsSource(),
    globus_uuid=SOCKEYE_SOURCE.endpoint,
)

agent=agent_local
# agent=agent_ssh
# agent=agent_slurm
agent.Deploy()

2025-03-16_18-11-27  | /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
2025-03-16_18-11-27  | /home/tony
2025-03-16_18-11-27  | >>> AGENT_HOME=/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
2025-03-16_18-11-27  | >>> mkdir -p $AGENT_HOME
2025-03-16_18-11-27  | >>> mkdir -p /home/tony/.globus
2025-03-16_18-11-27  | >>> mkdir -p /home/tony/.globusonline
2025-03-16_18-11-27  | >>> [ -e /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/metasmith.sif ] || apptainer pull /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-03-16_18-11-27  | staged [msm_stub]
2025-03-16_18-11-27  | staged [msm]
2025-03-16_18-11-27  | >>> cd /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home && ./msm api deploy_from_container
2025-03-16_18-11-27  | including dev binds
2025-03-16_18-11-28  | 2025-03-16_18-11-28  | api call to [deploy_from_container] 

In [3]:
CACHE = WORKSPACE_ROOT/"main/local_mock/cache/xgdb_tests"
trlib = TransformInstanceLibrary.Load("./transforms/simple_1")
xgdb = DataInstanceLibrary.Load(CACHE/"test.xgdb")
# refdb = DataInstanceLibrary.Load(CACHE/"ref.xgdb")
types = DataTypeLibrary.Load(WORKSPACE_ROOT/"main/local_mock/prototypes/metagenomics.dev3.yml")

In [4]:
# chinook_ep = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2F").endpoint
# refdb.SaveAs(GlobusSource(endpoint=chinook_ep, path="/Metasmith/ref.xgdb").AsSource())

refdb = DataInstanceLibrary.LoadFrom(
    src=GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fref.xgdb%2F").AsSource(),
    dest=CACHE/"ref.image.xgdb",
    as_image=True,
)
refdb.remote_src

Source(address='globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb', type=SourceType.GLOBUS)

In [5]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[trlib],
    targets=[
        types["orf_annotations"].WithLineage([
            types["contigs"],
            # xgdb["example.fna"].type,
        ]),
    ],
)

print(task.plan._key)
for step in task.plan.steps:
    print(step.transform.name)

NjapTxHs
pprodigal
diamond


In [ ]:
with open(WORKSPACE_ROOT/"secrets/slurm_account") as f:
    slurm_account = f.read().strip()
    
task.container_runtime = ContainerRuntime.APPTAINER
task.config = dict(
    nextflow = dict(
        preset = "default",
        # preset = "slurm",
        slurm_account=slurm_account,
    ),

)

In [7]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-03-16_18-11-29  | connecting to deployed agent
2025-03-16_18-11-29  | starting relay service


E| > 2025-03-16_18-11-29 E| relay server already running in [relay/connections]


 | > 2025-03-16_18-11-29  | connecting to relay as [JorGgdhvDygV]
W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/NjapTxHs]
W| clearing previously staged task
2025-03-16_18-11-31  | sending metadata for workflow [NjapTxHs]
2025-03-16_18-11-32  | staging
 | > including dev binds
 | > 2025-03-16_18-11-33  | api call to [stage_workflow] with [{'task_key': 'NjapTxHs'}]
 | > 2025-03-16_18-11-33  | staging workflow [NjapTxHs] with [2] data libs and [1] transform libs
 | > 2025-03-16_18-11-33  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
 | > 2025-03-16_18-11-33  | work [/ws/runs/NjapTxHs]
 | > 2025-03-16_18-11-33  | data [/msm_home/data]
 | > 2025-03-16_18-11-33  | external work [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/NjapTxHs]
 | > 2025-03-16_18-11-33  | external data [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/data]
 | > 2025-03-16_18-11-33  | 

In [8]:
# import shutil
# work_root = WORKSPACE_ROOT/"main/local_mock/cache/local_home/runs/kCvaS6w9"
# for p in [".nextflow", "nxf_logs", "nxf_work", "results"]:
#     shutil.rmtree(work_root/p, ignore_errors=True)
# shutil.rmtree(WORKSPACE_ROOT/"main/local_mock/mock/cache", ignore_errors=True)
    
agent.RunWorkflow(task)

2025-03-16_18-11-34  | connecting to deployed agent
2025-03-16_18-11-34  | starting relay service


E| > 2025-03-16_18-11-34 E| relay server already running in [relay/connections]


 | > 2025-03-16_18-11-34  | connecting to relay as [LRyfNuN4CfKC]
2025-03-16_18-11-35  | executing workflow
 | > including dev binds
 | > 2025-03-16_18-11-36  | api call to [execute_workflow] with [{'key': 'NjapTxHs'}]
 | > 2025-03-16_18-11-36  | workspace [/msm_home/runs/NjapTxHs]
 | > 2025-03-16_18-11-36  | external workspace [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/NjapTxHs]
 | > 2025-03-16_18-11-36  | executing workflow [NjapTxHs] with preset [slurm]
 | > 2025-03-16_18-11-36  | preset [slurm]
 | > 2025-03-16_18-11-36  | steps [2]
 | > 2025-03-16_18-11-36  | locating input data with personal globus endpoint
 | > 2025-03-16_18-11-36  | [xcVQmhPd71XZ] is at [/msm_home/runs/NjapTxHs/_metasmith/task/transforms/xcVQmhPd71XZ]
 | > 2025-03-16_18-11-36  | [dz4jLH96oFtR] is at [/msm_home/runs/NjapTxHs/_metasmith/task/data/dz4jLH96oFtR]
 | > 2025-03-16_18-11-36  | [Mmivz64sjqUn] is at [/msm_home/data/Mmivz64sjqUn]
 | > 2025-03-16_18-11-36  | calling nextflow

In [9]:
# import mimetypes

# def istext(filename):
#     s=open(filename, encoding="latin1").read(512)
#     text_characters = "".join([chr(x) for x in range(32, 127)] + list("\n\r\t\b"))
#     translation_table = str.maketrans("", "", text_characters)
#     if not s:
#         # Empty files are considered text
#         return True
#     if "\0" in s:
#         # Files with null bytes are likely binary
#         return False
#     # Get the non-text characters (maps a character to itself then
#     # use the 'remove' option to get rid of the text characters.)
#     t = s.translate(translation_table)
#     # If more than 30% non-text characters, then
#     # this is considered a binary file
#     if float(len(t))/float(len(s)) > 0.30:
#         return False
#     return True

# # istext("/home/tony/workspace/tools/Metasmith/metasmith.sif")
# istext("/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz/nxf_work/dd/6d7e3eef979ec8010613bb376b63b6/container.diamond.oci.uri")

In [10]:
# from metasmith.coms.containers import ContainerRuntime

# s = ContainerRuntime.APPTAINER.name
# ContainerRuntime[s], s